In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 1. データの読み込み
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')

target_col = 'class'

le = LabelEncoder()
X = train.drop(columns=['id', target_col], errors='ignore')
y = le.fit_transform(train[target_col])
X_test = test.drop(columns=['id'], errors='ignore')

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

num_classes = len(le.classes_)

# 2. 実験速度を最優先：手堅い 5-Fold に戻す
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_xgb = np.zeros((len(X), num_classes))
oof_preds_lgb = np.zeros((len(X), num_classes))
test_preds_xgb = np.zeros((len(X_test), num_classes))
test_preds_lgb = np.zeros((len(X_test), num_classes))

# 3. 学習ループ（爆速5回戦）
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    print(f"--- FOLD {fold+1} / 5 ---")
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    # --- ① XGBoost（パラメータをより深く、表現力重視にチューニング） ---
    model_xgb = xgb.XGBClassifier(
        n_estimators=1500,
        max_depth=8,              # 6から8に深化（より複雑なパターンを学習）
        learning_rate=0.03,       # 低学習率でじっくり最適化
        subsample=0.8,            # 行のサンプリングで過学習防止
        colsample_bytree=0.8,     # 列のサンプリングで多様性確保
        objective='multi:softprob',
        num_class=num_classes,
        tree_method='hist',
        device='cuda',
        enable_categorical=True,
        early_stopping_rounds=50,
        random_state=42
    )
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_preds_xgb[val_idx] = model_xgb.predict_proba(X_val)
    test_preds_xgb += model_xgb.predict_proba(X_test) / folds.n_splits
    
    # --- ② LightGBM（XGBの深さに合わせて葉の数を 63 に倍増） ---
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1500,
        max_depth=8,
        num_leaves=63,            # 決定木の表現力を一気に強化
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multiclass',
        num_class=num_classes,
        device='gpu',
        random_state=42,
        verbose=-1
    )
    
    # 警告ブロック裏技
    null_fds = [os.open(os.devnull, os.O_RDWR) for _ in range(2)]
    save_fds = [os.dup(1), os.dup(2)]
    os.dup2(null_fds[0], 1); os.dup2(null_fds[1], 2)
    try:
        model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    finally:
        os.dup2(save_fds[0], 1); os.dup2(save_fds[1], 2)
        os.close(null_fds[0]); os.close(null_fds[1])
        os.close(save_fds[0]); os.close(save_fds[1])
    
    oof_preds_lgb[val_idx] = model_lgb.predict_proba(X_val)
    test_preds_lgb += model_lgb.predict_proba(X_test) / folds.n_splits

# 4. 単体スコア確認
score_xgb = accuracy_score(y, np.argmax(oof_preds_xgb, axis=1))
score_lgb = accuracy_score(y, np.argmax(oof_preds_lgb, axis=1))
print(f"\n[チューニング後] XGBoost Accuracy: {score_xgb:.5f}")
print(f"[チューニング後] LightGBM Accuracy: {score_lgb:.5f}")


## =====================================================================
## 🔥 【新しく入れたのはココ！】上位者のスタッカー用のデータ書き出し
## =====================================================================
## 50:50に混ぜた「一番強いブレンド確率」を、上位者が読み込めるNumPy形式で保存します。
oof_preds_blend = (oof_preds_xgb * 0.5) + (oof_preds_lgb * 0.5)
test_preds_blend = (test_preds_xgb * 0.5) + (test_preds_lgb * 0.5)

np.save("my_sharp_blend_oof.npy", oof_preds_blend)    ## 学習データ用の予測確率
np.save("my_sharp_blend_test.npy", test_preds_blend)  ## 本番データ用の予測確率
print("【システム】スタッキング専用のファイルを保存しました！")
## =====================================================================


# 5. ブレンド（画面表示用）
score_blend = accuracy_score(y, np.argmax(oof_preds_blend, axis=1))
print(f"==================================================")
print(f"🔥 【鋭敏チューニング】ブレンド後 OOF Accuracy: {score_blend:.5f}")
print(f"==================================================")

# 6. 提出用ファイルの作成
sub = pd.DataFrame({'id': test['id']})
sub[target_col] = le.inverse_transform(np.argmax(test_preds_blend, axis=1))
sub.to_csv("submission.csv", index=False)
print("軽量・高火力版 'submission.csv' が生成されました！")